In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e9/train.csv
/kaggle/input/competitions/playground-series-s6e9/test.csv


In [16]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nMissing values in train:")
print(train.isnull().sum())

print("\nMissing values in test:")
print(test.isnull().sum())

print("\nTarget balance (Will_Buy_EV):")
print(train['Will_Buy_EV'].value_counts())
print(train['Will_Buy_EV'].value_counts(normalize=True))

Train shape: (668665, 15)
Test shape: (286571, 14)

Missing values in train:
id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64

Missing values in test:
id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible     

In [17]:
print(train.describe())
print("\n")
print(train.describe(include='object'))

                 id            Age  Annual_Income_USD  Daily_Commute_km  \
count  668665.00000  668665.000000      668665.000000     668665.000000   
mean   334332.00000      47.039171       84769.266989         32.158298   
std    193027.10321      12.875448       28648.029042         18.730474   
min         0.00000      25.000000       30000.000000          5.000000   
25%    167166.00000      36.000000       67376.000000         17.200000   
50%    334332.00000      47.000000       84880.000000         33.600000   
75%    501498.00000      58.000000      102753.000000         47.400000   
max    668664.00000      69.000000      188549.000000         98.700000   

       Number_of_Cars_Owned  Charging_Stations_Near_Home  \
count         668665.000000                668665.000000   
mean               1.712626                     4.960408   
std                0.729275                     3.926843   
min                1.000000                     0.000000   
25%                1.000

In [18]:
import matplotlib.pyplot as plt

train['Will_Buy_EV_num'] = (train['Will_Buy_EV'] == 'Yes').astype(int)

numeric_cols = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
                 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
                 'Environmental_Concern_Level']

for col in numeric_cols:
    print(train.groupby('Will_Buy_EV')[col].mean())
    print()

Will_Buy_EV
No     47.083293
Yes    46.830654
Name: Age, dtype: float64

Will_Buy_EV
No     81794.588556
Yes    98827.302948
Name: Annual_Income_USD, dtype: float64

Will_Buy_EV
No     32.553478
Yes    30.290714
Name: Daily_Commute_km, dtype: float64

Will_Buy_EV
No     1.711319
Yes    1.718802
Name: Number_of_Cars_Owned, dtype: float64

Will_Buy_EV
No     4.989947
Yes    4.820807
Name: Charging_Stations_Near_Home, dtype: float64

Will_Buy_EV
No     7.205490
Yes    7.038432
Name: Charging_Stations_Near_Work, dtype: float64

Will_Buy_EV
No     2.630395
Yes    4.377268
Name: Environmental_Concern_Level, dtype: float64



In [19]:
categorical_cols = ['Gender', 'City_Type', 'Current_Car_Type',
                     'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

for col in categorical_cols:
    print(train.groupby(col)['Will_Buy_EV_num'].mean())
    print()

Gender
Female    0.177641
Male      0.172253
Other     0.173732
Name: Will_Buy_EV_num, dtype: float64

City_Type
Rural       0.193389
Suburban    0.180936
Urban       0.161058
Name: Will_Buy_EV_num, dtype: float64

Current_Car_Type
Hatchback    0.174299
SUV          0.180953
Sedan        0.171967
Truck        0.156413
Name: Will_Buy_EV_num, dtype: float64

Home_Charging_Possible
No     0.127085
Yes    0.195819
Name: Will_Buy_EV_num, dtype: float64

Subsidy_Available
No     0.005757
Yes    0.274695
Name: Will_Buy_EV_num, dtype: float64

Range_Anxiety_Level
High      0.001367
Low       0.189027
Medium    0.041745
Name: Will_Buy_EV_num, dtype: float64



In [20]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['Gender', 'City_Type', 'Current_Car_Type',
            'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

train.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV,Will_Buy_EV_num
0,0,66,92887.0,23.4,2,3,7,1.0,1,1,2,1,0,1,No,0
1,1,38,30000.0,5.0,1,2,2,4.0,1,0,1,1,0,1,No,0
2,2,26,94389.0,36.8,1,8,15,5.0,0,2,2,0,1,1,Yes,1
3,3,66,73580.0,23.7,2,6,9,3.0,1,1,0,1,0,1,No,0
4,4,54,57898.0,50.8,1,2,3,3.0,1,1,0,1,0,1,No,0


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

feature_cols = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
                'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
                'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type',
                'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

X = train[feature_cols]
y = train['Will_Buy_EV_num']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

val_preds = model.predict_proba(X_val)[:, 1]
print("Validation ROC-AUC:", roc_auc_score(y_val, val_preds))

Validation ROC-AUC: 0.9171583023274361


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_val_preds = rf_model.predict_proba(X_val)[:, 1]
print("Random Forest Validation ROC-AUC:", roc_auc_score(y_val, rf_val_preds))

KeyboardInterrupt: 

In [23]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42)
lgb_model.fit(X_train, y_train)

lgb_val_preds = lgb_model.predict_proba(X_val)[:, 1]
print("LightGBM Validation ROC-AUC:", roc_auc_score(y_val, lgb_val_preds))

[LightGBM] [Info] Number of positive: 93586, number of negative: 441346
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174949 -> initscore=-1.550948
[LightGBM] [Info] Start training from score -1.550948
LightGBM Validation ROC-AUC: 0.9417952287173613


In [25]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42),
                             X, y, cv=5, scoring='roc_auc')

print("CV scores:", cv_scores)
print("Mean CV ROC-AUC:", cv_scores.mean())

[LightGBM] [Info] Number of positive: 93424, number of negative: 441508
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009837 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174646 -> initscore=-1.553048
[LightGBM] [Info] Start training from score -1.553048
[LightGBM] [Info] Number of positive: 93423, number of negative: 441509
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009997 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 13
[LightGBM] [Info

In [26]:
final_model = lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42)
final_model.fit(X, y)

test_preds = final_model.predict_proba(test[feature_cols])[:, 1]

submission = pd.DataFrame({
    'id': test['id'],
    'Will_Buy_EV': test_preds
})

submission.to_csv('submission.csv', index=False)
submission.head()

[LightGBM] [Info] Number of positive: 116779, number of negative: 551886
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014495 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 668665, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553058
[LightGBM] [Info] Start training from score -1.553058


,id,Will_Buy_EV
0,668665,0.010568
1,668666,0.019181
2,668667,0.006164
3,668668,0.003088
4,668669,0.018063
